In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

url = "https://www.zomato.com/bhagalpur/royal-darbar-bhagalpur-locality/order"
driver.get(url)

time.sleep(5)

In [13]:
read_more_buttons = driver.find_elements(
    By.XPATH,
    "//span[contains(text(),'Read More')]"
)
read_more_buttons

[]

In [19]:

last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height


# ----------------------------
# CLICK ALL "READ MORE"
# ----------------------------
read_more_buttons = driver.find_elements(
    By.XPATH, "//span[contains(text(),'Read More')]"
)

for btn in read_more_buttons:
    try:
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.2)
    except:
        pass


# ----------------------------
# EXTRACT MENU
# ----------------------------
menu_data = []

# category blocks (each section = category + dishes)
sections = driver.find_elements(By.XPATH, "//section")

current_category = None

for sec in sections:
    try:
        # category name (usually in p tag inside section header)
        cat = sec.find_element(By.CSS_SELECTOR, "p").text
        if cat and "(" in cat:
            current_category = cat.split("(")[0].strip()
    except:
        pass

    # find dishes inside this section
    dishes = sec.find_elements(By.XPATH, ".//div")

    for dish in dishes:
        try:
            name = dish.find_element(By.TAG_NAME, "h4").text
        except:
            continue

        try:
            price = dish.find_element(By.XPATH, ".//*[contains(text(),'₹')]").text
        except:
            price = ""

        try:
            desc = dish.find_element(By.TAG_NAME, "p").text
        except:
            desc = ""

        menu_data.append({
            "category": current_category,
            "dish": name,
            "description": desc,
            "price": price
        })


# ----------------------------
# SAVE TO CSV
# ----------------------------
df = pd.DataFrame(menu_data)
df.to_csv("zomato_menu.csv", index=False)

driver.quit()

print("Scraping completed!")
print(df.head())

MaxRetryError: HTTPConnectionPool(host='localhost', port=55021): Max retries exceeded with url: /session/f3ea17b4411a4e7df4d0b88ec86a8059/element/f.03ACB47DBFBFB2C46C33CDA52C206F71.d.97CC67AD1DF551431F91B56B19C5E391.e.26163/elements (Caused by NewConnectionError("HTTPConnection(host='localhost', port=55021): Failed to establish a new connection: [Errno 61] Connection refused"))